# 黄色いロボットのテクスチャをPaint3Dで調整する

既存のUVとベースカラーを持つGLBを1本選びます。
この版は、金属度と粗さの画像を持つ単一プリミティブのGLBに対応します。
この試行ではPaint3DのStage 2に含まれるUV Tileを、元のベースカラーに直接適用します。
Stage 1からの全面再生成は省き、黄色い塗装の外観をどこまで維持できるかを確かめます。

ColabでL4 GPUを選んで上から実行してください。
黄色いロボット1体でL4での実行と出力GLBを確認しました。
公開ノートブックは実行時にPaint3Dのコードとモデルの重みを取得します。
依存先のライセンスと確認範囲はImageToPlayableの `LICENSING.md` と `THIRD_PARTY_NOTICES.md` に記載しています。

In [ ]:
from pathlib import Path
import subprocess as sp

root = Path('/content/paint3d_robot')
root.mkdir(exist_ok=True)
repo = root / 'Paint3D'
prefix = root / 'env-py39'
commit = 'd5545d38db6aaf78efc563269f70177688a4218e'

def run(*args, **kwargs):
    return sp.run([str(arg) for arg in args], check=True, **kwargs)

run('nvidia-smi')
if not repo.exists():
    run('git', 'clone', 'https://github.com/OpenTexture/Paint3D.git', repo)
run('git', '-C', repo, 'checkout', '--detach', commit)
mamba = root / 'bin/micromamba'
if not mamba.exists():
    archive = root / 'micromamba.tar.bz2'
    run('curl', '-fLsS', '--retry', '3', 'https://micro.mamba.pm/api/micromamba/linux-64/latest', '-o', archive)
    run('tar', '-xjf', archive, '-C', root, 'bin/micromamba')
python = prefix / 'bin/python'
if not python.exists():
    run(mamba, 'create', '-y', '-p', prefix, '-c', 'conda-forge', 'python=3.9', 'pip')

run(python, '-m', 'pip', 'install', 'torch==1.12.1+cu116',
    '--extra-index-url', 'https://download.pytorch.org/whl/cu116')
run(python, '-c', 'import sys; assert sys.version_info[:2] == (3, 9); print(sys.version)')
run(python, '-m', 'pip', 'install', '--only-binary=kaolin', 'kaolin==0.13.0',
    '-f', 'https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-1.12.1_cu116.html')
run(python, '-m', 'pip', 'install', 'numpy==1.26.4', 'Pillow==10.4.0',
    'diffusers==0.25.0', 'transformers==4.38.2', 'accelerate==0.23.0',
    'huggingface-hub==0.25.2', 'omegaconf==2.3.0', 'loguru==0.7.2',
    'trimesh==3.20.2', 'imageio==2.34.2', 'opencv-python-headless==4.10.0.84',
    'safetensors==0.4.5')
run(python, '-c', 'import torch, kaolin; assert torch.cuda.is_available(); print(torch.__version__, kaolin.__version__, torch.cuda.get_device_name())')


In [ ]:
from google.colab import files
print('元のGLBを1本選びます。')
uploaded = files.upload()
names = [name for name in uploaded if name.lower().endswith('.glb')]
assert len(names) == 1, 'GLBを1本選択してください'
source = root / 'original.glb'
source.write_bytes(uploaded[names[0]])
del uploaded


In [ ]:
"""Export the original UV layout or replace only the base-color image."""
import copy
import io
import json
import struct
from pathlib import Path

import numpy as np
from PIL import Image


def read_glb(path):
    with Path(path).open('rb') as stream:
        magic, version, length = struct.unpack('<4sII', stream.read(12))
        assert (magic, version) == (b'glTF', 2)
        size, kind = struct.unpack('<I4s', stream.read(8))
        assert kind == b'JSON'
        document = json.loads(stream.read(size))
        size, kind = struct.unpack('<I4s', stream.read(8))
        assert kind == b'BIN\0'
        data = stream.read(size)
        assert stream.tell() == length and not stream.read(1)
    return document, data


def base_color_image(document):
    primitive, = [p for mesh in document['meshes'] for p in mesh['primitives']]
    material = document['materials'][primitive['material']]['pbrMetallicRoughness']
    texture = document['textures'][material['baseColorTexture']['index']]
    assert material['baseColorTexture'].get('texCoord', 0) == 0
    assert texture['source'] != document['textures'][material['metallicRoughnessTexture']['index']]['source']
    return primitive, texture['source']


def accessor(document, data, index):
    info = document['accessors'][index]
    view = document['bufferViews'][info['bufferView']]
    dtype = np.dtype({5123: '<u2', 5125: '<u4', 5126: '<f4'}[info['componentType']])
    width = {'VEC2': 2, 'VEC3': 3, 'SCALAR': 1}[info['type']]
    offset = view.get('byteOffset', 0) + info.get('byteOffset', 0)
    return np.ndarray((info['count'], width), dtype=dtype, buffer=data, offset=offset,
                      strides=(view.get('byteStride', dtype.itemsize * width), dtype.itemsize))


def export_inputs(source, folder):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    document, data = read_glb(source)
    primitive, image = base_color_image(document)
    attributes = primitive['attributes']
    vertices = accessor(document, data, attributes['POSITION'])
    uv = accessor(document, data, attributes['TEXCOORD_0']).copy()
    uv[:, 1] = 1 - uv[:, 1]  # glTF top-left to OBJ bottom-left
    assert primitive.get('mode', 4) == 4
    faces = accessor(document, data, primitive['indices']).reshape(-1, 3) + 1
    with (folder / 'robot.obj').open('w') as stream:
        np.savetxt(stream, vertices, fmt='v %.9g %.9g %.9g')
        np.savetxt(stream, uv, fmt='vt %.9g %.9g')
        np.savetxt(stream, np.repeat(faces, 2, axis=1), fmt='f %d/%d %d/%d %d/%d')
    view = document['bufferViews'][document['images'][image]['bufferView']]
    start = view.get('byteOffset', 0)
    (folder / 'original.png').write_bytes(data[start:start + view['byteLength']])
    return len(vertices), len(faces)


def replace_base_color(source, png_path, destination):
    document, data = read_glb(source)
    _, image = base_color_image(document)
    png = Path(png_path).read_bytes()
    with Image.open(io.BytesIO(png)) as texture:
        assert texture.format == 'PNG'
        texture.verify()
    modified = copy.deepcopy(document)
    modified['images'][image]['bufferView'] = len(document['bufferViews'])
    modified['images'][image]['mimeType'] = 'image/png'
    modified['bufferViews'].append({'buffer': 0, 'byteOffset': len(data), 'byteLength': len(png)})
    modified['buffers'][0]['byteLength'] = len(data) + len(png)
    binary = data + png + bytes(-len(png) % 4)
    header = json.dumps(modified, separators=(',', ':')).encode('utf-8')
    header += b' ' * (-len(header) % 4)
    with Path(destination).open('wb') as stream:
        stream.write(struct.pack('<4sIII4s', b'glTF', 2, 28 + len(header) + len(binary), len(header), b'JSON'))
        stream.write(header)
        stream.write(struct.pack('<I4s', len(binary), b'BIN\0'))
        stream.write(binary)
    check, check_data = read_glb(destination)
    assert check_data[:len(data)] == data
    assert modified == check

print('頂点数、三角形数', export_inputs(source, root))
with Image.open(root / 'original.png') as original:
    print('元のベースカラー解像度', original.size)


In [ ]:
import os
import subprocess
from pathlib import Path

runner = root / "run_paint3d.py"
log_path = root / "inference.log"

runner.write_text(r'''
import json
import subprocess
import sys
import time
from pathlib import Path

print("[1/4] Paint3DとCUDAを読み込みます")
import torch
from omegaconf import OmegaConf
from PIL import Image

from controlnet.diffusers_cnet_img2img import img2imgControlNet
from paint3d.config.train_config_paint3d import TrainConfig
from paint3d.models.textured_mesh import TexturedMeshModel
from paint3d.utils import save_tensor_image


SIZE = 1024  # L4向けの比較用解像度
PROMPT = (
    "UV texture atlas, yellow painted robot, satin yellow enamel, "
    "original yellow and dark panel colors, subtle manufacturing detail, diffuse albedo"
)
NEGATIVE_PROMPT = (
    "cast shadow, baked highlight, glare, changed pattern, "
    "lettering, logos, strong weathering"
)


@torch.no_grad()
def make_uv_position_map(obj_path, output_path):
    config = TrainConfig()
    config.log.exp_path = str(output_path.parent)
    config.log.cache_path = str(output_path.parent / "cache")
    config.guide.shape_path = str(obj_path)
    config.guide.texture_resolution = [SIZE, SIZE]
    config.guide.force_run_xatlas = False

    mesh = TexturedMeshModel(config, device=torch.device("cuda"))
    if mesh.mesh.vt is None or mesh.mesh.ft is None or mesh.mesh.ft.min().item() < 0:
        raise ValueError("OBJから元のUV座標を読み込めませんでした")
    position_map = mesh.UV_pos_render().permute(0, 3, 1, 2)
    save_tensor_image(position_map, str(output_path))


def make_tile_config(base_color, uv_position):
    config = OmegaConf.load("controlnet/config/UV_based_inpaint_template.yaml").img2img
    config.image_path = str(base_color)
    config.controlnet_units[0].condition_image_path = str(uv_position)
    config.controlnet_units[1].condition_image_path = str(base_color)
    config.prompt = PROMPT
    config.negative_prompt = NEGATIVE_PROMPT
    config.width = config.height = SIZE
    config.seed = 42
    config.denoising_strength = 0.35
    config.num_inference_steps = 30
    return config


def main(work_dir):
    start = time.monotonic()
    mesh = work_dir / "robot.obj"
    original = work_dir / "original.png"
    for path in (mesh, original):
        if not path.is_file():
            raise FileNotFoundError(f"先にGLBの展開セルを実行してください。見つからないファイル: {path}")

    print(f"[2/4] 元のUVから位置マップを作ります。GPU: {torch.cuda.get_device_name()}")
    uv_position = work_dir / "uv_position.png"
    make_uv_position_map(mesh, uv_position)
    torch.cuda.empty_cache()

    base_color = work_dir / "original_rgb.png"
    with Image.open(original) as image:
        image.convert("RGB").save(base_color)
    config = make_tile_config(base_color, uv_position)

    print("[3/4] UV Tileのモデルを取得して読み込みます")
    model = img2imgControlNet(config)
    # MultiControlNetの内側にある各モデルに直接設定する。
    model.pipe.unet.set_attention_slice("max")
    controlnets = model.pipe.controlnet.nets
    for controlnet in controlnets:
        controlnet.set_attention_slice("max")
    print(f"attention slicing=max をUNetとControlNet {len(controlnets)}本に適用")

    print(f"[4/4] UV Tileを {SIZE} × {SIZE}、attention slicing有効で実行します")
    result = model.infernece(config)[0].convert("RGB")
    report = {
        "gpu": torch.cuda.get_device_name(),
        "torch": torch.__version__,
        "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
        "elapsed_seconds_including_model_loading": round(time.monotonic() - start, 1),
        "attention_slice": "max",
        "img2img_config": OmegaConf.to_container(config, resolve=True),
    }
    result.save(work_dir / "paint3d.png")
    (work_dir / "run.json").write_text(json.dumps(report, indent=2))
    print(f"出力先: {work_dir / 'paint3d.png'}")


if __name__ == "__main__":
    main(Path(sys.argv[1]))
'''.lstrip(), encoding="utf-8")

command = [str(python), "-u", str(runner), str(root)]
with log_path.open("w", encoding="utf-8") as log, subprocess.Popen(
    command,
    cwd=repo,
    env={**os.environ, "PYTHONPATH": str(repo)},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()

if process.returncode != 0:
    raise RuntimeError(
        f"Paint3Dの推論が終了コード {process.returncode} で停止しました。"
        f"上の出力に原因が表示されています。ログ: {log_path}"
    )


In [ ]:
from zipfile import ZipFile
from IPython.display import display
result = root / 'robot_paint3d.glb'
replace_base_color(source, root / 'paint3d.png', result)
display(Image.open(root / 'paint3d.png').resize((512, 512)))
bundle = root / 'robot_paint3d_result.zip'
with ZipFile(bundle, 'w') as archive:
    for name in ['robot_paint3d.glb', 'paint3d.png', 'original.png', 'uv_position.png', 'run.json']:
        archive.write(root / name, name)
files.download(str(bundle))


元のGLBと出力の `robot_paint3d.glb` をGLB対応のビューアで開きます。
正面と背面で黄色と暗色パネルの境界を確認し、光源の向きと再生位置を揃えて見比べます。